In [0]:
from pyspark.sql import functions as F

LANDING    = "/Volumes/claims/bronze/landing"
CHECKPOINT = "/Volumes/claims/bronze/landing/_checkpoints"

TABLES = ["beneficiary", "inpatient", "outpatient", "carrier"]

def ingest(name: str):
    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/{name}/schema")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .load(f"{LANDING}/{name}")
        .select("*",
                F.col("_metadata.file_name").alias("_source_file"),
                F.current_timestamp().alias("_ingested_at"))
        .writeStream
        .option("checkpointLocation", f"{CHECKPOINT}/{name}/write")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"claims.bronze.{name}")
        .awaitTermination())

for t in TABLES:
    print(f"ingesting {t} ...")
    ingest(t)
    print("  done")

In [0]:
for t in TABLES:
    cols = spark.table(f"claims.bronze.{t}").columns
    print(f"\n=== {t} ({len(cols)} columns) ===")
    for c in cols:
        print(" ", c)

In [0]:
%sql
SELECT 'beneficiary' t, COUNT(*) c FROM claims.bronze.beneficiary
UNION ALL SELECT 'inpatient',  COUNT(*) FROM claims.bronze.inpatient
UNION ALL SELECT 'outpatient', COUNT(*) FROM claims.bronze.outpatient
UNION ALL SELECT 'carrier',    COUNT(*) FROM claims.bronze.carrier;